# Deforestation Issue Analysis Using Support Vector Machine (SVM)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import seaborn as sns

df = pd.read_csv('deforestation_dataset.csv')
df.head()


## Data Inspection and Cleaning

In [ ]:

print(df.info())
print(df.isnull().sum())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(exclude='object').columns:
    df[col] = df[col].fillna(df[col].median())


## Encoding and Feature Scaling

In [ ]:

le = LabelEncoder()
df['Country'] = le.fit_transform(df['Country'])

target = 'Forest_Loss_Area_km2'

X = df.drop(columns=[target])
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Train-Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)


## Linear SVM Model

In [ ]:

svm_linear = SVR(kernel='linear', C=1.0)

svm_linear.fit(X_train, y_train)

pred = svm_linear.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

print('MAE:', mae)
print('MSE:', mse)
print('RMSE:', rmse)
print('R2:', r2)


## Hyperparameter Tuning

In [ ]:

param_grid = {
    'kernel':['linear','rbf','poly'],
    'C':[0.1,1,10,100],
    'gamma':['scale','auto']
}

grid = GridSearchCV(
    SVR(),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('Best Parameters:', grid.best_params_)
print('Best CV Score:', grid.best_score_)

best_model = grid.best_estimator_


## Cross Validation

In [ ]:

scores = cross_val_score(
    best_model,
    X_scaled,
    y,
    cv=5,
    scoring='r2'
)

print('CV Scores:', scores)
print('Mean CV Score:', scores.mean())


## Feature Importance Analysis

In [ ]:

result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean
}).sort_values('Importance', ascending=False)

print(importance.head(10))

plt.figure(figsize=(10,5))
plt.barh(
    importance['Feature'][:10],
    importance['Importance'][:10]
)
plt.title('Top Feature Importance')
plt.show()


## Correlation Heatmap

In [ ]:

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## Actual vs Predicted

In [ ]:

best_pred = best_model.predict(X_test)

plt.figure(figsize=(6,5))
plt.scatter(y_test, best_pred)
plt.xlabel('Actual Forest Loss')
plt.ylabel('Predicted Forest Loss')
plt.title('Actual vs Predicted')
plt.show()



# Final Report

## Key Contributors to Deforestation
- Population growth
- GDP and economic expansion
- Agriculture land percentage
- Illegal lumbering incidents
- Policy strictness and protected areas

## Interpretation
Higher agricultural land expansion and illegal logging generally contribute to increased forest loss.
Strong policy enforcement and larger protected areas can reduce deforestation impacts.

## Recommendations
1. Strengthen anti-illegal logging regulations.
2. Increase protected forest coverage.
3. Promote sustainable agriculture.
4. Improve transparency and reduce corruption.
5. Use international aid for conservation programs.

## Evaluation Metrics
- MAE
- MSE
- RMSE
- R² Score
- Cross-validation R²
